# TinyBERT Reproduction — Colab Runner

Runs the full task-specific distillation pipeline (`run_experiment.py`) on a Colab GPU.

**Before you start:** set the runtime to GPU via `Runtime → Change runtime type → T4 GPU` (or better).

Pipeline steps (handled by `run_experiment.py`):
1. Fine-tune BERT-base teacher on the chosen GLUE task
2. Data augmentation (BERT MLM + optional GloVe)
3. Two-phase task-specific distillation into TinyBERT-4L
4. Teacher vs. student comparison + retention %

In [ ]:
import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    print('\n>>> No GPU detected. Go to Runtime > Change runtime type > GPU, then re-run.')
!nvidia-smi

torch: 2.11.0+cu128
CUDA available: True
Sat May 30 17:13:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   29C    P0             44W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+------

## 2. Get the code

Clones the repo from GitHub over HTTPS. This works as long as the repo is **public**.

If you keep it private, you'll instead need a GitHub personal access token (fine-grained, with **Contents: Read** on this repo) and clone with `https://<TOKEN>@github.com/vanjus98/tinybert-reproduction.git`.

In [6]:
import os, subprocess

REPO_URL = 'https://github.com/vanjus98/tinybert-reproduction.git'
REPO_DIR = 'tinybert-reproduction'

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)

os.chdir(REPO_DIR)
print('cwd:', os.getcwd())
print(os.listdir('.'))

cwd: /content/tinybert-reproduction/tinybert-reproduction
['README.md', 'Group Project Details.pdf', 'run_experiment.py', 'main.py', 'requirements.txt', 'tinybert', '.gitignore', '.git', '1909.10351v5.pdf']


## 3. Install dependencies

We intentionally do **not** reinstall `torch`/`numpy` — Colab ships them with CUDA support already, and reinstalling can break GPU access. We only add the NLP-specific packages.

In [3]:
!pip install -r requirements.txt

In [ ]:
!pip install -q -U "transformers>=4.30" "datasets>=2.14" "scikit-learn>=1.3" "scipy>=1.11"

In [4]:
# Sanity check that the package imports
import transformers, datasets, sklearn, scipy
print('transformers', transformers.__version__)
print('datasets', datasets.__version__)
from tinybert.glue_utils import TASK_CONFIG
print('available tasks:', list(TASK_CONFIG.keys()))

transformers 5.0.0
datasets 4.0.0
available tasks: ['sst2', 'mrpc', 'qqp', 'qnli', 'rte', 'cola', 'stsb', 'mnli']


## 4. Smoke test (fast)

Runs the entire pipeline end-to-end with reduced epochs / small augmentation on a 5% subset. Use this first to confirm everything works (~a few minutes on a T4) before launching the full run.

In [34]:
!python -u run_experiment.py --task sst2 --fast --subset 0.01


Step 1: Fine-tuning BERT-base teacher on SST2
Device: cuda
  Teacher using 673/67349 training examples (1% subset)
Loading weights: 100% 199/199 [00:00<00:00, 4674.41it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you

In [35]:
!python -u run_experiment.py --task mrpc --fast --subset 0.01



Step 1: Fine-tuning BERT-base teacher on MRPC
Device: cuda
mrpc/train-00000-of-00001.parquet: 100% 649k/649k [00:01<00:00, 587kB/s]  
mrpc/validation-00000-of-00001.parquet: 100% 75.7k/75.7k [00:00<00:00, 287kB/s]
mrpc/test-00000-of-00001.parquet: 100% 308k/308k [00:00<00:00, 667kB/s] 
Generating train split: 100% 3668/3668 [00:00<00:00, 383009.04 examples/s]
Generating validation split: 100% 408/408 [00:00<00:00, 163258.54 examples/s]
Generating test split: 100% 1725/1725 [00:00<00:00, 411416.72 examples/s]
  Teacher using 36/3668 training examples (1% subset)
Loading weights: 100% 199/199 [00:00<00:00, 6307.61it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED |

## 5. Full run

Edit the flags below as needed:
- `--task` : `sst2`, `mrpc`, `qqp`, `qnli`, `rte`, `cola`, `stsb`, `mnli`
- `--skip_augmentation` : train on original data only (much faster)
- `--na 20` : augmented samples per example (paper default 20)
- `--subset 0.3` : use a fraction of training data when compute is limited
- `--glove_path glove.6B.100d.txt` : enable GloVe multi-piece augmentation (download first)

Full augmentation (`Na=20`) on the entire SST-2 train set is heavy. If you hit time/memory limits, start with `--subset 0.3` or `--skip_augmentation`.

In [7]:
!python -u run_experiment.py --task sst2


Step 1: Fine-tuning BERT-base teacher on SST2
Device: cuda
README.md: 100% 35.3k/35.3k [00:00<00:00, 48.1MB/s]
sst2/train-00000-of-00001.parquet: 100% 3.11M/3.11M [00:00<00:00, 3.28MB/s]
sst2/validation-00000-of-00001.parquet: 100% 72.8k/72.8k [00:00<00:00, 422kB/s]
sst2/test-00000-of-00001.parquet: 100% 148k/148k [00:00<00:00, 948kB/s]
Generating train split: 100% 67349/67349 [00:00<00:00, 1095622.59 examples/s]
Generating validation split: 100% 872/872 [00:00<00:00, 346643.26 examples/s]
Generating test split: 100% 1821/1821 [00:00<00:00, 571394.30 examples/s]
config.json: 100% 570/570 [00:00<00:00, 3.00MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 222kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 9.24MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 10.4MB/s]
model.safetensors: 100% 440M/440M [00:01<00:00, 331MB/s]
Loading weights: 100% 199/199 [00:00<00:00, 924.56it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassific

In [8]:
!python -u run_experiment.py --task sst2 --subset 0.2



Step 1: Using existing teacher at checkpoints/sst2/teacher

Step 2: Loading GLUE/sst2 dataset
  Using 13469/67349 training examples (20% subset)

Step 3: Data augmentation (Na=20)
Loading weights: 100% 202/202 [00:00<00:00, 941.26it/s, Materializing param=cls.predictions.transform.dense.weight]                 
BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
  Augmenting 13469 training examples ...
Augmenting: 100% 13469/13469 [14:41<00:00, 15.28sent/s]
  Augmented dataset size: 282849 (was 13469)

Step 4: Task-specific distillation
Device: cuda
Loading weights: 100% 201/201 [00:00<00:

## 6. Inspect results

In [9]:
import json, glob
for path in glob.glob('checkpoints/*/results.json'):
    print('==', path, '==')
    with open(path) as f:
        print(json.dumps(json.load(f), indent=2))

== checkpoints/sst2/results.json ==
{
  "task": "sst2",
  "teacher": {
    "accuracy": 0.9288990825688074
  },
  "student": {
    "accuracy": 0.9059633027522935
  }
}


In [12]:
# Save checkpoints to your local machine
import shutil
from google.colab import files

# zip the whole checkpoints/ folder (teacher + student for every task run so far)
shutil.make_archive('checkpoints', 'zip', 'checkpoints')
print('Zipped. Size:')
!du -h checkpoints.zip

files.download('checkpoints.zip')   # triggers browser download to ~/Downloads


Zipped. Size:
439M	checkpoints.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. (Optional) Save checkpoints to Google Drive

Colab wipes local files when the runtime disconnects. Mount Drive and copy the outputs to keep them.

In [11]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p '/content/drive/MyDrive/tinybert_checkpoints'
!cp -r checkpoints/* '/content/drive/MyDrive/tinybert_checkpoints/'
print('Copied checkpoints to Drive.')

MessageError: [dfs_ephemeral] Credentials propagation unsuccessful

In [13]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
dest = '/content/drive/MyDrive/tinybert_checkpoints'
os.makedirs(dest, exist_ok=True)
shutil.copy('checkpoints.zip', dest)
print('Copied to Drive:', dest)
!ls -lh '{dest}/checkpoints.zip'

Mounted at /content/drive
Copied to Drive: /content/drive/MyDrive/tinybert_checkpoints
-rw------- 1 root root 439M May 30 21:03 /content/drive/MyDrive/tinybert_checkpoints/checkpoints.zip
